# Author: José Ángel de Bustos Pérez
# License: GNU General Public License v3.0

# Basic implementation for BFV algorithm using Pyphel (https://pyfhel.readthedocs.io/)

BFV (Brakerski–Fan–Vercauteren) is a homomorphic encryption scheme based on the RLWE (Ring Learning With Errors) problem, which is resistant to quantum attacks.

This scheme works with integers modulo t exactly.

The first step is to set the parameters and generate the keys:

* **n**, polynomial degree, a power of two. The higher the polynomial degree, the greater the security and the more slots for batching, but it will also require a larger number of operations and therefore be slower.
* **t_bits**, range of values that can be encrypted: [-2^(t_bits - 1), 2^(t_bits - 1)]
* **sec**, security level.

We can operate on elements individually or operate on all of them at once (vectorization), which makes FHE usable for real-world problems. This property is referred to as **batching**.

Slots are the number of elements the scheme can act on simultaneously; in other words, they indicate the level of parallelism we can achieve.

When operating with homomorphic algorithms, noise is introduced; when that noise is contained it does not affect the operation (the data can still be decrypted), but when the noise is not contained it will prevent decryption and therefore further operations. The **noise budget** indicates the tolerance we have before reaching noise levels that would make the scheme unusable.

In [1]:
from Pyfhel import Pyfhel
import numpy as np

HE = Pyfhel()

bfv_params = {
    'scheme': 'BFV',
    'n': 2**13,        # 8192 coefficients => 8192 slots for batching
    't_bits': 20,      # plaintext modulus of approximately 20 bits
    'sec': 128,        # 128 security bits
}

HE.contextGen(**bfv_params)
HE.keyGen()           # Key creation
HE.rotateKeyGen()     # Required for rotations (we will use it in batching)
HE.relinKeyGen()      # Required to reduce size after multiplication

print(f"Scheme: {HE.scheme}")
print(f"n (polinomial degree): {HE.n}")
print(f"t (plaintext modulus): {HE.t}")
print(f"Batching slot number: {HE.n}")
print(f"Initial noise budget(bits): {HE.noise_level(HE.encrypt(0))}")


Scheme: Scheme_t.bfv
n (polinomial degree): 8192
t (plaintext modulus): 1032193
Batching slot number: 8192
Initial noise budget(bits): 146


## Basic operations

We will perform basic operations such as addition and multiplications to check how noise is increasing. We will start with addition:

In [2]:
from random import randrange

# Randon number generation
a = randrange(100)
b = randrange(100)

# Encrypting data
fhe_a = HE.encryptInt(np.array([a], dtype=np.int64))
fhe_b = HE.encryptInt(np.array([b], dtype=np.int64))

print(f"Plaintext data: a = {a}, b = {b}")
print(f"Noise budget for fhe_a after encrypting a = {a}: {HE.noise_level(fhe_a)} bits")
print(f"Noise budget for fhe_b after encrypting b = {b}: {HE.noise_level(fhe_b)} bits")

# Homomorphic operation (addition)
fhe_suma = fhe_a + fhe_b

# Getting the operation value, decrypting
suma = HE.decryptInt(fhe_suma)[0]

print(f"\nHomomorphic addition: {a} + {b} = {suma}")
print(f"Noise budget after addition: {HE.noise_level(fhe_suma)} bits ")

Plaintext data: a = 49, b = 21
Noise budget for fhe_a after encrypting a = 49: 146 bits
Noise budget for fhe_b after encrypting b = 21: 146 bits

Homomorphic addition: 49 + 21 = 70
Noise budget after addition: 146 bits 


Addition using homomorphic encryption hardly add noise.

We are going to proceed with multiplication.

The BFV algorithm internally operates using polynomials. Polynomial multiplication result in a higher degree polynomial and for that reason the algorithm must use an operation known as **realinarization** to come back to the original polynomial degree.

In [3]:
# Homomorphic multiplication
fhe_mult = fhe_a * fhe_b
# Realinarization
~fhe_mult

# Getting the operation value, decrypting
mult = HE.decryptInt(fhe_mult)[0]

print(f"\nHomomorphic multiplication: {a} * {b} = {mult}")
print(f"Noise budget after multiplication: {HE.noise_level(fhe_mult)} bits ")


Homomorphic multiplication: 49 * 21 = 1029
Noise budget after multiplication: 114 bits 


**Noise budget** has been increased from 146 bits to 114 bits. Homomorphic multiplication adds more noise than homomorphic addition. This means that after a number of operations we will not be able to perform more homomorphic multiplications due to data will be corrupted.

## Performing homomorphic encryption with multiple operations

We are going to show how to perform more complex operations with homomorphic encryption. Let's assume we want to perform homomorphic encryption to:

$$f(x,y) = (x+y)^2 - (x-y)^2$$

In [4]:
# Random number generation
x = randrange(100)
y = randrange(100)

expected_value = (x + y)**2 - (x-y)**2

# Encrypting data
fhe_x = HE.encryptInt(np.array([x], dtype=np.int64))
fhe_y = HE.encryptInt(np.array([y], dtype=np.int64))

# (x + y)
fhe_addition_xy = fhe_x + fhe_y

# (x + y)^2
fhe_square_add = fhe_addition_xy * fhe_addition_xy
# Realinarization
~fhe_square_add

# (x - y)
fhe_substraction_xy = fhe_x - fhe_y

# (x - y)^2
fhe_square_substraction = fhe_substraction_xy * fhe_substraction_xy
# Realinarization
~fhe_square_substraction

# Final
fhe_final = fhe_square_add - fhe_square_substraction

# Getting the operation value, decrypting
final = HE.decryptInt(fhe_final)[0]

print(f"Operation: ({x} + {y})**2 - ({x}-{y})**2")
print(f"Expected value: {expected_value}")
print(f"Homomorphic value: {final}")
print(f"Noise budget after operation: {HE.noise_level(fhe_final)} bits")

Operation: (44 + 72)**2 - (44-72)**2
Expected value: 12672
Homomorphic value: 12672
Noise budget after operation: 113 bits


Added noise is mostly the same that the noise added by the homomorphic multiplication. Now, we will perform the following:

$$f(x,y) = (x+y)^2 - (x-y)^3$$

In [5]:
expected_value_third = (x + y)**2 - (x-y)**3

# (x - y)^3
fhe_third_substraction = fhe_square_substraction * fhe_substraction_xy
# Realinarization
~fhe_third_substraction

# Final
fhe_final = fhe_square_add - fhe_third_substraction

# Getting the operation value, decrypting
final = HE.decryptInt(fhe_final)[0]

print(f"Operation: ({x} + {y})**2 - ({x}-{y})**3")
print(f"Expected value: {expected_value_third}")
print(f"Homomorphic value: {final}")
print(f"Noise budget after operation: {HE.noise_level(fhe_final)} bits")

Operation: (44 + 72)**2 - (44-72)**3
Expected value: 35408
Homomorphic value: 35408
Noise budget after operation: 81 bits


Performing an additional homomorphic multiplication increases the noise reducing the **Noise budget**.

## Batching operation

Batching allows us to perform the same homomorphic operation to several data at the same time, parallelism.

If we are familiar with processor architectures we will know what **SIMD** (**S**imple **I**nstruction **M**ultiple **D**ata) is. One single operation using only one clock cicle operate on multiple data. Batching is the same but used in homomorphic encryption and it is used to speed up operations.

In [6]:
# Randon number generation
size = 10 # number of integer to operate at the same time
v1 = np.random.choice(np.arange(100, dtype=np.int64), size=(size), replace=False)
v2 = np.random.choice(np.arange(100, dtype=np.int64), size=(size), replace=False)

# Encrypting data
fhe_v1 = HE.encryptInt(v1)
fhe_v2 = HE.encryptInt(v2)

# Item by item homomorphic addition
fhe_add_vectorial = fhe_v1 + fhe_v2

# Getting the operation value, decrypting
final = HE.decryptInt(fhe_add_vectorial)[:size]

print(f"v1: {v1}")
print(f"v2: {v2}")
print(f"Noise budget for fhe_v1 after encrypting v1: {HE.noise_level(fhe_v1)} bits")
print(f"Noise budget for fhe_v2 after encrypting v2: {HE.noise_level(fhe_v2)} bits")
print(f"Homomorphic v1 + v2:  {final}")
print(f"Expected value: {v1 + v2}")
print(f"Noise budget after operation: {HE.noise_level(fhe_add_vectorial)} bits")

v1: [42 91 55 24 18  9 44 39  5 68]
v2: [25 46 30 88  1 49 59 80 47 55]
Noise budget for fhe_v1 after encrypting v1: 146 bits
Noise budget for fhe_v2 after encrypting v2: 146 bits
Homomorphic v1 + v2:  [ 67 137  85 112  19  58 103 119  52 123]
Expected value: [ 67 137  85 112  19  58 103 119  52 123]
Noise budget after operation: 146 bits


As expected this operation does not increases noise.

Let's see what happens with batching multiplication:

In [7]:
# Item by item homomorphic multiplication
fhe_multiplication_vectorial = fhe_v1 * fhe_v2
# Realinarization
~fhe_multiplication_vectorial

# Getting the operation value, decrypting
final = HE.decryptInt(fhe_multiplication_vectorial)[:size]

print(f"\nHomomorphic v1 * v2  = {final}")
print(f"Expected value = {v1 * v2}")
print(f"Noise budget after operation: {HE.noise_level(fhe_multiplication_vectorial)} bits")


Homomorphic v1 * v2  = [1050 4186 1650 2112   18  441 2596 3120  235 3740]
Expected value = [1050 4186 1650 2112   18  441 2596 3120  235 3740]
Noise budget after operation: 114 bits


As expected this operation increases noise.

## Dot product or scalar product

Dot product or scalar product is a fundamental operation used in multiple algorithms such as linear regresion, support vector machines (SVM) or neural network algorithms. For this reason been able to operate it using homomorphic encription will ease using such algoritms with homomorphic encryption.

In [8]:
# Random sample
size = 10 # random sample size
v1 = np.random.choice(np.arange(100, dtype=np.int64), size=(size), replace=False)
v2 = np.random.choice(np.arange(100, dtype=np.int64), size=(size), replace=False)

expected_value = int(np.dot(v1, v2))

# Encrypt data
fhe_v1 = HE.encryptInt(v1)
fhe_v2 = HE.encryptInt(v2)

# Cdot operation with encrypted data
fhe_cdot = fhe_v1 * fhe_v2
# Relinearization
~fhe_cdot

# vector with product, component to component
value_cdot = HE.decryptInt(fhe_cdot)[:size]

print(f"v1: {v1}")
print(f"v2: {v2}")
# cdot product is the sum for all the values in value_cdot
print(f"Homomorphic v1 * v2: {np.sum(value_cdot)}")
print(f"Expected value: {expected_value}")
print(f"Noise budget: {HE.noise_level(fhe_cdot)} bits")

v1: [83 93 80 43 52 37 90 62 48 87]
v2: [44 79  5 74 68 94 70 71 41 46]
Homomorphic v1 * v2: 38267
Expected value: 38267
Noise budget: 114 bits


## How many operations can be done before data is corrupted?

We have seen that the multiplication operation adds noise to the data. For this reason is important to know how many operations can be done without corrupting data. This number  of operations is the operational limit which indicates the maximum number of operations that can be performed by the algorithm.

In [9]:
data = randrange(10)
fhe_data = HE.encryptInt(np.array([data], dtype=np.int64))

print(f"Encrypting data = {data}")
print(f"Initial noise budget: {HE.noise_level(fhe_data)} bits\n")

# Start multiplying encrypted data
for i in range(1, 15):
    fhe_data = fhe_data * fhe_data   
    # Relinearization
    ~fhe_data
    expected_value = data ** (2 ** i)
    fhe_value = HE.decryptInt(fhe_data)[0]
    noise = HE.noise_level(fhe_data)
    ok = "OK " if fhe_value == expected_value else "FAIL"
    print(f"Iteration {i:2d}: {data}^{2**i:<8} | "
          f"Expected value: {expected_value:<12} Encrypted value: {fhe_value:<12} | "
          f"Noise budget: {noise:3d} bits | {ok}")
    if noise == 0:
        print("\n>>> Noise budget empty: data corruption.")
        break

Encrypting data = 1
Initial noise budget: 146 bits

Iteration  1: 1^2        | Expected value: 1            Encrypted value: 1            | Noise budget: 114 bits | OK 
Iteration  2: 1^4        | Expected value: 1            Encrypted value: 1            | Noise budget:  81 bits | OK 
Iteration  3: 1^8        | Expected value: 1            Encrypted value: 1            | Noise budget:  48 bits | OK 
Iteration  4: 1^16       | Expected value: 1            Encrypted value: 1            | Noise budget:  15 bits | OK 
Iteration  5: 1^32       | Expected value: 1            Encrypted value: 31731        | Noise budget:   0 bits | FAIL

>>> Noise budget empty: data corruption.
